# 🦀 Day 3: Custom Error Types

---

In [ ]:
use std::fmt;
use std::num::ParseIntError;

// Define an error enum
#[derive(Debug)]
enum AppError {
    NotFound(String),
    InvalidInput(String),
    ParseError(ParseIntError),
    IoError(std::io::Error),
}

impl fmt::Display for AppError {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        match self {
            AppError::NotFound(item) => write!(f, "Not found: {}", item),
            AppError::InvalidInput(msg) => write!(f, "Invalid input: {}", msg),
            AppError::ParseError(e) => write!(f, "Parse error: {}", e),
            AppError::IoError(e) => write!(f, "IO error: {}", e),
        }
    }
}

impl std::error::Error for AppError {
    fn source(&self) -> Option<&(dyn std::error::Error + 'static)> {
        match self {
            AppError::ParseError(e) => Some(e),
            AppError::IoError(e) => Some(e),
            _ => None,
        }
    }
}

// From implementations for ? operator
impl From<ParseIntError> for AppError {
    fn from(e: ParseIntError) -> Self { AppError::ParseError(e) }
}

impl From<std::io::Error> for AppError {
    fn from(e: std::io::Error) -> Self { AppError::IoError(e) }
}

println!("Custom AppError defined! ✅");

In [ ]:
// Using custom errors with ?
fn validate_and_parse(input: &str) -> Result<u32, AppError> {
    if input.is_empty() {
        return Err(AppError::InvalidInput("Empty input".into()));
    }
    let number: u32 = input.parse()?;  // ParseIntError → AppError automatically!
    if number == 0 {
        return Err(AppError::InvalidInput("Zero not allowed".into()));
    }
    Ok(number)
}

for input in ["42", "", "abc", "0"] {
    match validate_and_parse(input) {
        Ok(n) => println!("✅ '{}' → {}", input, n),
        Err(e) => println!("❌ '{}' → {}", input, e),
    }
}

In [ ]:
// Error context: wrapping errors with additional info
#[derive(Debug)]
struct ContextError {
    context: String,
    source: Box<dyn std::error::Error>,
}

impl fmt::Display for ContextError {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        write!(f, "{}: {}", self.context, self.source)
    }
}

impl std::error::Error for ContextError {
    fn source(&self) -> Option<&(dyn std::error::Error + 'static)> {
        Some(self.source.as_ref())
    }
}

// Helper trait to add context to any Result
trait WithContext<T> {
    fn with_context(self, ctx: &str) -> Result<T, ContextError>;
}

impl<T, E: std::error::Error + 'static> WithContext<T> for Result<T, E> {
    fn with_context(self, ctx: &str) -> Result<T, ContextError> {
        self.map_err(|e| ContextError {
            context: ctx.to_string(),
            source: Box::new(e),
        })
    }
}

// Usage
let result = "abc".parse::<i32>().with_context("parsing user age");
println!("{:?}", result);

## 🏋️ Exercises

In [ ]:
// Exercise: Create a ConfigError enum with variants:
// MissingField(String), InvalidValue { field: String, value: String },
// FileError(io::Error)
// Implement Display, Error, and From<io::Error>

// YOUR CODE HERE


## 🎯 Key Takeaways

1. Custom error enums give **meaningful, domain-specific** errors
2. Implement `Display` and `Error` for your error types
3. `From<E>` enables `?` operator to auto-convert errors
4. `source()` chains errors for debugging
5. Error context adds "what was happening" to "what went wrong"

---
**Tomorrow:** `thiserror` and `anyhow` crates! 📦